# DEC5 paired source-graph and full-block TSDF controls
Unit: frame 000973, three held camera anchors F/J/L per variant. Seven diagnostic variants produce 21 renders; four new candidate variants have 12 failed anchor verdicts. These are not 21 temporal frames. Face PSNR/SSIM/LPIPS use only F and the same GT-only polygon. No room/full-frame metrics.
Question: do false image-grid couplings across depth discontinuities, or insufficient TSDF resolution/truncation, explain the remaining skin/tube defect? Neither tested change is sufficient. These checks distinguish comparison integrity from reconstruction quality. No variant is promoted to every-40th-frame or continuous fly-through validation.

In [1]:
from pathlib import Path
import hashlib, json, math
root=Path('/mnt/data/lookcloser_dec5_5a3_surface_repair')
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for b in iter(lambda:stream.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
audits=[json.loads((root/name/'findings.json').read_text()) for name in ['depth_separated_source_control','full_block_narrow_control']]
checked={}
for audit in audits:
    assert audit['audit_status']=='comparison_integrity_pass_candidate_quality_fail'
    assert audit['visual_pass']==0 and audit['visual_fail']==6 and not audit['temporal_promotion']
    for path,expected in audit['retained_hashes'].items():
        assert digest(path)==expected,path
        checked[path]=expected
assert audits[0]['off_control_byte_identical_F_J_L']
assert audits[1]['depth_maps_verified']==62 and audits[1]['depth_shape']==[1080,1920]
print('Verified retained hashes:',len(checked))
print('Off-control F/J/L reproduces historical baseline byte-for-byte; original 62 depth maps remain unchanged.')


Verified retained hashes: 176
Off-control F/J/L reproduces historical baseline byte-for-byte; original 62 depth maps remain unchanged.


In [2]:
rows=[r for audit in audits for r in audit['rows']]
assert len(rows)==7 and len({r['variant'] for r in rows})==7
print('Variant | Face PSNR | Face SSIM | Face LPIPS | Mesh triangles/components if changed')
for row in rows:
    v=[row[k] for k in ['face_psnr','face_ssim','face_lpips']]
    assert all(math.isfinite(x) for x in v)
    print(row['variant'],*[round(x,6) for x in v],str(row.get('triangles','unchanged'))+'/'+str(row.get('components','unchanged')),sep=' | ')
assert all(r['unchanged_label_rgb_max_error']==0 for r in audits[0]['paired'])
print('Graph change only: every unchanged source label retains identical RGB.')
print('Source-graph changed-label counts:',[(r['rank'],r['anchor'],r['changed_labels']) for r in audits[0]['paired']])


Variant | Face PSNR | Face SSIM | Face LPIPS | Mesh triangles/components if changed
off_rank001 | 28.724285 | 0.889098 | 0.047989 | unchanged/unchanged
depth_rank001 | 28.721514 | 0.889168 | 0.047968 | unchanged/unchanged
off_rank0 | 28.821976 | 0.913522 | 0.062508 | unchanged/unchanged
depth_rank0 | 28.806019 | 0.913621 | 0.062285 | unchanged/unchanged
fine_wide | 28.748835 | 0.889802 | 0.047824 | 681257/1
fine_narrow | 28.755987 | 0.891535 | 0.048329 | 603889/2
finer_narrow | 28.766863 | 0.89186 | 0.048132 | 2649029/6
Graph change only: every unchanged source label retains identical RGB.
Source-graph changed-label counts: [('rank001', 'F', 12289), ('rank001', 'J', 5931), ('rank001', 'L', 5223), ('rank0', 'F', 142635), ('rank0', 'J', 212363), ('rank0', 'L', 88096)]


## Findings and next step
The depth-aware graph removes 2,856/2,546/2,255 source-label edges for F/J/L, but does not remove the skin patch or wrong tube contour. Zero camera-rank preference weakens the neck patch while softening face details and exposing an incorrect skin-colored tube rim.
Full-block truncation .0015 with voxel .00025, then .000125, preserves the same source-camera color response to isolate geometry. The finer mesh has 2,649,029 triangles but six connected components and still fails native inspection. Geometry and appearance causes must be checked against original train observations at the specific remaining contour; a larger triangle count is not evidence of a repair.
Confidence is high in rejection of these exact candidates from actual crops, but not in a universal causal claim about every depth-aware graph or every TSDF setting. The detailed native verdicts and standalone audit scripts are retained beside the referenced findings. All fit/prediction stages exclude F/J/L RGB, and original calibration, source data and published campaign are unchanged.